# 03 – Task 2: Node Classification (community vs. non-community)

**Question:** Can a classifier correctly label nodes as community members
using only graph-structural features (without knowing the community directly)?

## Binary classification target
- `1` – node belongs to an overlapping community
- `0` – non-community node

## Feature sets tested
We compare many feature sets to understand what information is most useful:

| Feature set | Description |
|-------------|-------------|
| `lcc` | Local clustering coefficient (single feature) |
| `bc` | Betweenness centrality |
| `cc` | Closeness centrality |
| `dc` | Degree centrality |
| `ndc` | Normalised degree centrality |
| `ec` | Eigenvector centrality |
| `eccen` | Eccentricity |
| `core` | Core number (k-core) |
| `participation + in_mod_deg` | Community-aware baseline ("cheating") |
| `node2vec` | 16-dim node2vec embeddings |
| `struc2vec` | 16-dim struc2vec embeddings |

## Models
Logistic Regression, Random Forest, LightGBM, XGBoost

## Metrics
- **ROC-AUC** – area under ROC curve
- **Average Precision** – area under precision-recall curve
- **Average Precision at cutoff** – precision-recall at thresholds 0.01, 0.05, 0.1

In [1]:
# ── CHANGE THIS to switch dataset ──────────────────────────────────────────
DATASET = "facebook"
# ───────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb

ROOT = Path("..")
PATH_COMBINED = ROOT / f"data/04_models/{DATASET}_combined.parquet"

# Thresholds for top-k precision evaluation
CUTOFFS = [0.01, 0.05, 0.1]

print(f"Dataset: {DATASET}")

Dataset: facebook


## Step 1 – Load data and standardise features

Run `01_data_preparation.ipynb` first.

We standardise all features (zero mean, unit variance) before training —
this is important for Logistic Regression and helps embeddings-based models too.

In [2]:
df = pd.read_parquet(PATH_COMBINED)

COMMUNITY_FREE  = ["lcc", "bc", "cc", "dc", "ndc", "ec", "eccen", "core"]
COMMUNITY_AWARE = ["participation", "in_mod_deg"]   # cheating baseline
N2V_COLS = [c for c in df.columns if c.startswith("emb_node2vec")]
S2V_COLS = [c for c in df.columns if c.startswith("emb_struc2vec")]
ALL_FEATURES = COMMUNITY_FREE + COMMUNITY_AWARE + N2V_COLS + S2V_COLS

# Standardise features
scaler = StandardScaler()
df[ALL_FEATURES] = scaler.fit_transform(df[ALL_FEATURES])

print(f"Nodes: {len(df)}")
print(f"Class distribution:\n{df['target'].value_counts().to_string()}")

Nodes: 22470
Class distribution:
target
0.0    16702
1.0     5768


## Step 2 – Stratified train / test split

Stratified = same class ratio in train and test.
70% train, 30% test.

In [3]:
X = df[ALL_FEATURES]
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=2023, stratify=y
)
X_train = X_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)

print(f"Train: {len(X_train)} | Test: {len(X_test)}")
print(f"Train class ratio: {y_train.mean():.3f}")
print(f"Test  class ratio: {y_test.mean():.3f}")

Train: 15729 | Test: 6741
Train class ratio: 0.257
Test  class ratio: 0.257


## Step 3 – Define classifiers

In [4]:
def make_classifiers():
    """Return dict of {name: unfitted classifier}."""
    return {
        "LogisticRegression": LogisticRegression(
            penalty="l2", C=1.0, class_weight="balanced",
            solver="lbfgs", max_iter=1000,
            fit_intercept=True, random_state=2023, n_jobs=-1
        ),

        "RandomForest": RandomForestClassifier(
            n_estimators=1000, criterion="gini",
            max_depth=10, min_samples_split=2,
            min_samples_leaf=1, n_jobs=-1, random_state=2023
        ),

        "LightGBM": lgb.LGBMClassifier(
            learning_rate=0.1, objective="binary",
            boosting_type="gbdt",
            n_estimators=1000,
            n_jobs=-1,
        ),

        "XGBoost": xgb.XGBClassifier(
            booster="gbtree", random_state=2023,
            objective="binary:logistic",
            base_score=0.5, verbosity=0,
            n_jobs=-1, eval_metric="auc",
        ),
    }

print("Classifiers:", list(make_classifiers().keys()))

Classifiers: ['LogisticRegression', 'RandomForest', 'LightGBM', 'XGBoost']


## Step 4 – Define feature sets to evaluate

In [5]:
# Each entry: (label_for_results, list_of_feature_columns)
FEATURE_SETS = (
    # Individual community-free features
    [(feat, [feat]) for feat in COMMUNITY_FREE]
    # Community-aware baseline ("cheating")
    + [("participation+in_mod_deg", COMMUNITY_AWARE)]
    # Embedding sets
    + [("node2vec", N2V_COLS)]
    + [("struc2vec", S2V_COLS)]
)

for label, cols in FEATURE_SETS:
    print(f"  {label:30s} → {len(cols)} column(s)")

  lcc                            → 1 column(s)
  bc                             → 1 column(s)
  cc                             → 1 column(s)
  dc                             → 1 column(s)
  ndc                            → 1 column(s)
  ec                             → 1 column(s)
  eccen                          → 1 column(s)
  core                           → 1 column(s)
  participation+in_mod_deg       → 2 column(s)
  node2vec                       → 16 column(s)
  struc2vec                      → 16 column(s)


## Step 5 – Train and evaluate all combinations

For each feature set × classifier we compute ROC-AUC and Average Precision.
We also compute Average Precision after thresholding predictions at 0.01 / 0.05 / 0.1.

In [6]:
records = []

for feat_label, feat_cols in FEATURE_SETS:
    for clf_name, clf in make_classifiers().items():
        try:
            clf.fit(X_train[feat_cols], y_train)
            y_pred_proba = clf.predict_proba(X_test[feat_cols])[:, 1]

            roc_auc = roc_auc_score(y_test, y_pred_proba)
            avg_prec = average_precision_score(y_test, y_pred_proba)

            row = {
                "features"      : feat_label,
                "model"         : clf_name,
                "roc_auc"       : round(roc_auc, 4),
                "avg_precision" : round(avg_prec, 4),
            }
            # Average Precision at cutoff: binarise predictions at threshold
            for cutoff in CUTOFFS:
                y_binary = (y_pred_proba > cutoff).astype(int)
                row[f"avg_prec@{cutoff}"] = round(
                    average_precision_score(y_test, y_binary), 4
                )

            records.append(row)
            print(f"  {feat_label:30s} | {clf_name:20s} | "
                  f"AUC={roc_auc:.3f}  AP={avg_prec:.3f}")

        except Exception as e:
            records.append({
                "features": feat_label, "model": clf_name,
                "roc_auc": None, "avg_precision": None, "error": str(e)
            })
            print(f"  ERROR {feat_label} | {clf_name}: {e}")

results = pd.DataFrame(records)
print(f"\nDone. {len(results)} combinations evaluated.")

/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  lcc                            | LogisticRegression   | AUC=0.546  AP=0.269
  lcc                            | RandomForest         | AUC=0.574  AP=0.293
[LightGBM] [Info] Number of positive: 4038, number of negative: 11691
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000142 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 15729, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256723 -> initscore=-1.063070
[LightGBM] [Info] Start training from score -1.063070
  lcc                            | LightGBM             | AUC=0.559  AP=0.285
  lcc                            | XGBoost              | AUC=0.569  AP=0.293
  bc                             | LogisticRegression   | AUC=0.550  AP=0.271


/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  bc                             | RandomForest         | AUC=0.586  AP=0.326
[LightGBM] [Info] Number of positive: 4038, number of negative: 11691
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000122 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 255
[LightGBM] [Info] Number of data points in the train set: 15729, number of used features: 1
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256723 -> initscore=-1.063070
[LightGBM] [Info] Start training from score -1.063070
  bc                             | LightGBM             | AUC=0.575  AP=0.315
  bc                             | XGBoost              | AUC=0.577  AP=0.313
  cc                             | LogisticRegression   | AUC=0.529  AP=0.247


/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


...

  participation+in_mod_deg       | RandomForest         | AUC=0.774  AP=0.515
[LightGBM] [Info] Number of positive: 4038, number of negative: 11691
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000151 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 482
[LightGBM] [Info] Number of data points in the train set: 15729, number of used features: 2
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256723 -> initscore=-1.063070
[LightGBM] [Info] Start training from score -1.063070
  participation+in_mod_deg       | LightGBM             | AUC=0.780  AP=0.508
  participation+in_mod_deg       | XGBoost              | AUC=0.779  AP=0.513
  node2vec                       | LogisticRegression   | AUC=0.951  AP=0.862


/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  node2vec                       | RandomForest         | AUC=0.984  AP=0.965
[LightGBM] [Info] Number of positive: 4038, number of negative: 11691
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000337 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 15729, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256723 -> initscore=-1.063070
[LightGBM] [Info] Start training from score -1.063070
  node2vec                       | LightGBM             | AUC=0.991  AP=0.979
  node2vec                       | XGBoost              | AUC=0.990  AP=0.974
  struc2vec                      | LogisticRegression   | AUC=0.532  AP=0.275


/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/seba/Desktop/BetaStar/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1184: FutureWarning: 'n_jobs' has no effect since 1.8 and will be removed in 1.10. You provided 'n_jobs=-1', please leave it unspecified.
  warnings.warn(msg, category=FutureWarning)


  struc2vec                      | RandomForest         | AUC=0.653  AP=0.414
[LightGBM] [Info] Number of positive: 4038, number of negative: 11691
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000329 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4080
[LightGBM] [Info] Number of data points in the train set: 15729, number of used features: 16
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.256723 -> initscore=-1.063070
[LightGBM] [Info] Start training from score -1.063070
  struc2vec                      | LightGBM             | AUC=0.640  AP=0.385
  struc2vec                      | XGBoost              | AUC=0.637  AP=0.383

Done. 44 combinations evaluated.


## Step 6 – Results

In [7]:
# Full results sorted by ROC-AUC
results.sort_values("roc_auc", ascending=False)

,features,model,roc_auc,avg_precision,avg_prec@0.01,avg_prec@0.05,avg_prec@0.1
38,node2vec,LightGBM,0.9913,0.9788,0.8660,0.8870,0.8900
39,node2vec,XGBoost,0.9896,0.9736,0.6616,0.7900,0.8391
37,node2vec,RandomForest,0.9843,0.9650,0.2785,0.4342,0.5785
36,node2vec,LogisticRegression,0.9510,0.8621,0.3395,0.4251,0.4838
34,participation+in_mod_deg,LightGBM,0.7805,0.5080,0.2941,0.3469,0.3650
35,participation+in_mod_deg,XGBoost,0.7792,0.5133,0.2638,0.3043,0.3411
33,participation+in_mod_deg,RandomForest,0.7743,0.5148,0.2650,0.2796,0.3049
21,ec,RandomForest,0.6791,0.3825,0.2747,0.2783,0.2835
22,ec,LightGBM,0.6769,0.3770,0.2682,0.2837,0.2945
23,ec,XGBoost,0.6759,0.3725,0.2694,0.2817,0.2960


In [8]:
# Best model per feature set (by ROC-AUC)
best = (
    results.sort_values("roc_auc", ascending=False)
           .groupby("features")
           .first()[["model", "roc_auc", "avg_precision"]]
           .sort_values("roc_auc", ascending=False)
)
print("Best model per feature set:")
best

Best model per feature set:


,model,roc_auc,avg_precision
features,,,
node2vec,LightGBM,0.9913,0.9788
participation+in_mod_deg,LightGBM,0.7805,0.5080
ec,RandomForest,0.6791,0.3825
struc2vec,RandomForest,0.6530,0.4143
cc,RandomForest,0.6525,0.3734
ndc,RandomForest,0.6153,0.3392
eccen,LightGBM,0.6102,0.3130
bc,RandomForest,0.5865,0.3256
core,LightGBM,0.5821,0.3190


In [9]:
# Pivot: feature sets as rows, models as columns (ROC-AUC)
pivot_auc = results.pivot_table(
    index="features", columns="model", values="roc_auc"
).round(4)
print("ROC-AUC by feature set and model:")
pivot_auc

ROC-AUC by feature set and model:


model,LightGBM,LogisticRegression,RandomForest,XGBoost
features,,,,
bc,0.5749,0.5502,0.5865,0.5767
cc,0.6407,0.5293,0.6525,0.6349
core,0.5821,0.4772,0.5821,0.5821
dc,0.5218,0.4863,0.5247,0.5248
ec,0.6769,0.5159,0.6791,0.6759
eccen,0.6102,0.5665,0.6102,0.6102
lcc,0.5592,0.5456,0.5736,0.5688
ndc,0.6031,0.5891,0.6153,0.6040
node2vec,0.9913,0.9510,0.9843,0.9896


In [10]:
# Save results
PATH_OUT = ROOT / f"data/05_reporting/{DATASET}_task2.parquet"
results.to_parquet(PATH_OUT, index=False)
print(f"Saved to {PATH_OUT}")

Saved to ../data/05_reporting/facebook_task2.parquet
